In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
import ast
import re
import matplotlib.gridspec as gridspec
from prophet import Prophet
import joblib

from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colors as mcolors
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter
from itertools import product

%matplotlib inline


In [ ]:
def train_prophet(data, duration):
    df_all_amount = data[['Description', 'YearMonth', 'LEVEL_1', 'Pendapatan in IDR']].copy()
    cust_type_unique = df_all_amount['Description'].unique()
    prod_level1_unique = df_all_amount['LEVEL_1'].unique()

    combinations = pd.DataFrame(
        product(cust_type_unique, prod_level1_unique),
        columns=['Description', 'LEVEL_1']
    )
    start_forecast = df_all_amount['YearMonth'].max() + pd.DateOffset(months=1)
    forecast_dates = pd.date_range(start=start_forecast, periods=duration, freq='MS')
    df_forecast_tmp = pd.DataFrame({'YearMonth': forecast_dates})

    df_forecast_tmp = df_forecast_tmp.merge(pd.DataFrame(combinations), how='cross')


    df_all_amount.rename(columns={'Pendapatan in IDR': 'y', 'YearMonth': 'ds'}, inplace=True)
    df_forecast_tmp.rename(columns={'Pendapatan in IDR': 'y', 'YearMonth': 'ds'}, inplace=True)


    model = Prophet()
    regressor=[]
    if(len(cust_type_unique)>1):
        regressor.append('Description')
    if(len(prod_level1_unique)>1):
        regressor.append('LEVEL_1')

    for i in regressor:
        model.add_regressor(i)
    
    target_col_train=['ds','y',]+regressor
    target_col_forecast=['ds']+regressor
    model.fit(df_all_amount[target_col_train])

    # pred = model.predict(df_all_amount[target_col_forecast])
    pred_forecast = model.predict(df_forecast_tmp[target_col_forecast])


    # df_all_amount['yhat'] = pred['yhat']
    # df_forecast_tmp['yhat'] = pred_forecast['yhat']

    # df_all_amount['yhat'] = pred['yhat'].apply(lambda x: 0 if x < 0 else x)
    df_forecast_tmp['yhat'] = pred_forecast['yhat'].apply(lambda x: 0 if x < 0 else x)


    # combined_forecast = pd.concat([df_all_amount[['ds','yhat','Description', 'LEVEL_1']],
    #                                df_forecast_tmp[['ds','yhat','Description', 'LEVEL_1']]], ignore_index=True)
    combined_forecast = df_forecast_tmp[['ds','yhat','Description', 'LEVEL_1']]
    combined_forecast['Method'] = 'PROPHET'
    combined_forecast.rename(columns={'yhat': 'Amount_Predict'}, inplace=True)
    model_data = {
        'model': model,
        'scalers_numerical_train': None,
        'scalers_categorical_train': None,
        'scalers_log_col': None,
        'coefficients_df_sorted': None
    }
    results=combined_forecast[['ds','Description', 'LEVEL_1','Amount_Predict','Method']].merge(df_all_amount,on=['ds','Description', 'LEVEL_1'],how='left')  
    results.rename(columns={'ds':'YearMonth','y':'Pendapatan in IDR'}, inplace=True)

    return results[['YearMonth','Description', 'LEVEL_1','Pendapatan in IDR','Amount_Predict','Method']], model_data

In [ ]:
def predict_cluster(new_data, centroids):
    distances = np.linalg.norm(centroids.values - new_data, axis=1)
    predicted_cluster = centroids.index[np.argmin(distances)]
    return predicted_cluster

In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])
data_pendapatan.drop(columns=['Kontribusi in IDR','Kontribusi in USD','Pendapatan in USD'],inplace=True)
cust_desc=pd.read_excel('./../data/customer_description_mapping.xlsx').drop(columns=['Unnamed: 0'])
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
data_pendapatan['YearMonth'] = data_pendapatan['Tanggal Nota'].dt.to_period('M')

# K-Means

In [ ]:
centroids=pd.read_excel('./../results/file/centroid_kmeans_rfm+descriptions.xlsx')
loaded_scaler = joblib.load('./../results/scaler_kmeans.pkl')

In [ ]:
customer_data = data_pendapatan.groupby('Cust No.')['Tanggal Nota'].max().reset_index()
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
most_recent_date = data_pendapatan['Tanggal Nota'].max()
customer_data['Tanggal Nota'] = pd.to_datetime(customer_data['Tanggal Nota'])
customer_data['Recency'] = (most_recent_date - customer_data['Tanggal Nota']).dt.days
customer_data.drop(columns=['Tanggal Nota'],inplace=True)
data_pendapatan_used=data_pendapatan.loc[data_pendapatan['Pendapatan in IDR']>0]
fm_feature = data_pendapatan_used.groupby(['Cust No.']).agg(
    freq_transaction=('Pendapatan in IDR', 'count'),
    total_transaction=('Pendapatan in IDR','sum'),
).reset_index()
customer_data = customer_data.merge(fm_feature, on='Cust No.', how='left')
customer_data.fillna(0,inplace=True)
customer_data_used = customer_data.loc[customer_data['freq_transaction'] != 0]
merged_data = customer_data_used.merge(cust_desc, how='left', right_on='Customer', left_on='Cust No.')
merged_data['Customer'].fillna(0,inplace=True)
merged_data['Customer']=merged_data['Customer'].astype('int64')
merged_data['Description'].fillna('Unknown', inplace=True)
merged_data.drop(columns=['Customer'],inplace=True)


In [ ]:
label_encoder = LabelEncoder()
label_encoder_level_1 = LabelEncoder()

In [ ]:
merged_data['Description'] = label_encoder.fit_transform(merged_data['Description'])
merged_data

In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])
data_pendapatan.drop(columns=['Kontribusi in IDR','Kontribusi in USD','Pendapatan in USD'],inplace=True)
cust_desc=pd.read_excel('./../data/customer_description_mapping.xlsx').drop(columns=['Unnamed: 0'])
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
data_pendapatan['YearMonth'] = data_pendapatan['Tanggal Nota'].dt.to_period('M')

In [ ]:
data_pendapatan_used=data_pendapatan.loc[data_pendapatan['Pendapatan in IDR']>0]
customer_data = data_pendapatan_used.groupby(['Cust No.','YearMonth','LEVEL_1'])[['Pendapatan in IDR']].sum()
customer_data.reset_index(inplace=True)
merged_data = customer_data.merge(cust_desc, how='left', right_on='Customer', left_on='Cust No.')
merged_data['Customer'].fillna(0,inplace=True)
merged_data['Customer']=merged_data['Customer'].astype('int64')
merged_data['Description'].fillna('Unknown', inplace=True)
merged_data.drop(columns=['Customer'],inplace=True)
agg_type_cust = merged_data.groupby(['Description','YearMonth','LEVEL_1'])[['Pendapatan in IDR']].sum()
agg_type_cust.reset_index(inplace=True)

In [ ]:
agg_type_cust_labeled=agg_type_cust.copy()

In [ ]:
label_encoder = LabelEncoder()
agg_type_cust_labeled['Description'] = label_encoder.fit_transform(agg_type_cust_labeled['Description'])
label_encoder_level_1 = LabelEncoder()
agg_type_cust_labeled['LEVEL_1'] = label_encoder_level_1.fit_transform(agg_type_cust_labeled['LEVEL_1'])

In [ ]:
agg_type_cust_labeled

In [ ]:
# input : recency, freq_transaction, total_transaction,description
# prophet -> take  prod level 1
# kmeans take prod type 